In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd drive/MyDrive/graph-fake-news-detection

/content/drive/MyDrive/graph-fake-news-detection


In [3]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.5 MB/s eta 0:00:00


In [8]:
import networkx as nx
import matplotlib.pyplot as plt

from torch_geometric.datasets import UPFD
from torch_geometric.utils import to_networkx


DATASET = "politifact"
FEATURE = "bert"

dataset = UPFD("./data", DATASET, FEATURE)
print(dataset)

UPFD(62, name=politifact, feature=bert)


In [21]:
print(dataset)
print(dataset.num_features)
print(dataset.num_classes)
print(len(dataset))

UPFD(62, name=politifact, feature=bert)
768
2
62


In [15]:
from pathlib import Path
import numpy as np

RAW_DIR = Path("data/politifact/raw")
edges = np.loadtxt(
    RAW_DIR / "A.txt",
    delimiter=",",
    dtype=int
)

print("shape:", edges.shape)
print(edges[:20])

shape: (40740, 2)
[[ 0  1]
 [ 0  2]
 [ 0  3]
 [ 0  4]
 [ 0  5]
 [ 0  6]
 [ 0  7]
 [ 0  8]
 [ 0  9]
 [ 0 10]
 [ 0 11]
 [ 0 12]
 [ 0 13]
 [ 0 14]
 [ 0 15]
 [ 0 16]
 [ 0 17]
 [ 0 18]
 [ 0 19]
 [ 0 20]]


In [16]:
node_graph_id = np.load(
    RAW_DIR / "node_graph_id.npy"
)

print(node_graph_id.shape)
print(node_graph_id[:30])

(41054,)
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [17]:
graph_labels = np.load(
    RAW_DIR / "graph_labels.npy"
)

print(graph_labels.shape)
print(graph_labels[:20])

(314,)
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [25]:
for graph_id in range(5):

    node_ids = np.where(
        node_graph_id == graph_id
    )[0]

    label = graph_labels[graph_id]

    print(
        f"Graph {graph_id}: "
        f"nodes={len(node_ids)}, "
        f"label={label}"
    )

Graph 0: nodes=497, label=0.0
Graph 1: nodes=21, label=0.0
Graph 2: nodes=142, label=0.0
Graph 3: nodes=15, label=0.0
Graph 4: nodes=30, label=0.0


In [26]:
graph_id = 0

node_ids = np.where(
    node_graph_id == graph_id
)[0]

node_set = set(node_ids)

mask = np.array([
    src in node_set and dst in node_set
    for src, dst in edges
])

graph_edges = edges[mask]

print("Nodes:", len(node_ids))
print("Edges:", len(graph_edges))
print(graph_edges[:20])

Nodes: 497
Edges: 496
[[ 0  1]
 [ 0  2]
 [ 0  3]
 [ 0  4]
 [ 0  5]
 [ 0  6]
 [ 0  7]
 [ 0  8]
 [ 0  9]
 [ 0 10]
 [ 0 11]
 [ 0 12]
 [ 0 13]
 [ 0 14]
 [ 0 15]
 [ 0 16]
 [ 0 17]
 [ 0 18]
 [ 0 19]
 [ 0 20]]


In [32]:
train = UPFD("./data", DATASET, FEATURE, split="train")
val = UPFD("./data", DATASET, FEATURE, split="val")
test = UPFD("./data", DATASET, FEATURE, split="test")

print(len(train), len(val), len(test))

62 31 221
